# PS S6E7 — NN: 주기 임베딩(PLR-lite) MLP · GPU

트리와 상보적인 유일한 모델 계열 (커뮤니티: 트리간 예측일치 98~99%, NN만 실질 블렌드 가중치 확보).
- V2와 동일 피처(서수+규칙) + 평균대치/지시자 + 표준화
- 수치 피처별 sin/cos 주기 임베딩 → GELU MLP, class-weighted CE
- 층화 5-fold × 2시드, fold별 best-epoch 스냅샷 (patience 3)
- 출력: submission.csv + **OOF/test 확률 npy** (로컬 블렌드 재료)
- 로컬 스모크(30k, CPU): OOF 0.9326 검증 완료

In [ ]:
import numpy as np, pandas as pd, time, glob, os, torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE=='cuda' else '')
_c = sorted(glob.glob('/kaggle/input/**/train.csv', recursive=True))
assert _c, 'competition data not mounted'
DATA = os.path.dirname(_c[0]); print('DATA =', DATA)
CLASSES = ['at-risk','unhealthy','fit']; N_CLASS = 3
train = pd.read_csv(f'{DATA}/train.csv'); test = pd.read_csv(f'{DATA}/test.csv')
y = train['health_condition'].map({c:i for i,c in enumerate(CLASSES)}).to_numpy()

In [ ]:
NUM = ['sleep_duration','heart_rate','bmi','calorie_expenditure',
       'step_count','exercise_duration','water_intake']
ORDINAL = {
    'stress_level':{'low':0,'medium':1,'high':2}, 'sleep_quality':{'poor':0,'average':1,'good':2},
    'physical_activity_level':{'sedentary':0,'moderate':1,'active':2},
    'smoking_alcohol':{'no':0,'occasional':1,'yes':2},
    'diet_type':{'veg':0,'balanced':1,'non-veg':2}, 'gender':{'female':0,'male':1,'other':2}}

def encode(df):
    X = df[NUM + list(ORDINAL)].copy()
    for c, m in ORDINAL.items(): X[c] = X[c].map(m).astype('float64')
    s, st, act = X['sleep_duration'], X['stress_level'], X['physical_activity_level']
    X['sleep_lt6'] = np.where(s.isna(), np.nan, (s < 6).astype(float))
    X['sleep_lt7'] = np.where(s.isna(), np.nan, (s < 7).astype(float))
    rp = np.full(len(X), np.nan)
    known = ~(s.isna() | st.isna() | act.isna())
    sv, stv, av = s[known], st[known], act[known]
    r = np.zeros(known.sum()); r[(sv<6)&(stv==2)] = 1.0; r[(sv>=7)&(stv==0)&(av==2)] = 2.0
    rp[known.to_numpy()] = r
    X['rule_pred'] = rp
    X['miss_sleep'] = s.isna().astype(float); X['miss_stress'] = st.isna().astype(float)
    X['miss_activity'] = act.isna().astype(float)
    return X

X, X_test = encode(train), encode(test)
for c in list(X.columns):
    if X[c].isna().any() or X_test[c].isna().any():
        X[f'na_{c}'] = X[c].isna().astype(float); X_test[f'na_{c}'] = X_test[c].isna().astype(float)
        mu = X[c].mean(); X[c] = X[c].fillna(mu); X_test[c] = X_test[c].fillna(mu)
print(X.shape, X_test.shape)

In [ ]:
class PeriodicEmbed(nn.Module):
    def __init__(self, n_feats, n_freq=8, sigma=1.0):
        super().__init__(); self.freq = nn.Parameter(torch.randn(n_feats, n_freq) * sigma)
    def forward(self, x):
        v = 2 * torch.pi * x.unsqueeze(-1) * self.freq
        return torch.cat([torch.sin(v), torch.cos(v)], -1).flatten(1)

class Net(nn.Module):
    def __init__(self, n_feats, n_freq=8, hidden=256):
        super().__init__(); self.pe = PeriodicEmbed(n_feats, n_freq)
        d_in = n_feats * 2 * n_freq + n_feats
        self.mlp = nn.Sequential(nn.Linear(d_in, hidden), nn.GELU(), nn.Dropout(0.15),
                                 nn.Linear(hidden, hidden//2), nn.GELU(), nn.Dropout(0.15),
                                 nn.Linear(hidden//2, N_CLASS))
    def forward(self, x): return self.mlp(torch.cat([self.pe(x), x], 1))

def train_fold(X_tr, y_tr, X_va, y_va, X_te, seed, epochs=15, bs=8192, lr=2e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    model = Net(X_tr.shape[1]).to(DEVICE)
    cw = compute_class_weight('balanced', classes=np.arange(N_CLASS), y=y_tr)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    dl = DataLoader(TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr)),
                    batch_size=bs, shuffle=True, num_workers=2)
    Xva_t = torch.tensor(X_va, dtype=torch.float32).to(DEVICE)
    Xte_t = torch.tensor(X_te, dtype=torch.float32).to(DEVICE)
    best_ba, best_va, best_te, patience = -1, None, None, 0
    for ep in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        sch.step(); model.eval()
        with torch.no_grad(): pv = torch.softmax(model(Xva_t), 1).cpu().numpy()
        ba = balanced_accuracy_score(y_va, pv.argmax(1))
        if ba > best_ba:
            best_ba, best_va, patience = ba, pv, 0
            with torch.no_grad(): best_te = torch.softmax(model(Xte_t), 1).cpu().numpy()
        else:
            patience += 1
            if patience >= 3: break
    return best_va, best_te, best_ba

In [ ]:
t0 = time.time(); SEEDS = [42, 777]
oof = np.zeros((len(X), N_CLASS)); test_pred = np.zeros((len(X_test), N_CLASS))
cols = X.columns
for si, SD in enumerate(SEEDS):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 폴드 고정(블렌드 정합)
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        sc = StandardScaler().fit(X.iloc[tr])
        pv, pt, ba = train_fold(sc.transform(X.iloc[tr]), y[tr], sc.transform(X.iloc[va]), y[va],
                                sc.transform(X_test[cols]), SD*100 + fold)
        oof[va] += pv / len(SEEDS); test_pred += pt / (5 * len(SEEDS))
        print(f'seed{SD} fold{fold} bal_acc={ba:.5f} ({time.time()-t0:.0f}s)')
cv = balanced_accuracy_score(y, oof.argmax(1))
print(f'[NN PLR x{len(SEEDS)}seed] OOF balanced_accuracy = {cv:.5f} total {time.time()-t0:.0f}s')

In [ ]:
np.save('nn_oof.npy', oof); np.save('nn_test.npy', test_pred)   # 로컬 블렌드 재료
inv = {i: c for i, c in enumerate(CLASSES)}
sub = pd.DataFrame({'id': test['id'], 'health_condition': [inv[i] for i in test_pred.argmax(1)]})
sub.to_csv('submission.csv', index=False)
ss = pd.read_csv(f'{DATA}/sample_submission.csv')
assert len(sub) == len(ss) and sub['id'].tolist() == ss['id'].tolist()
print('saved | sanity OK |', sub['health_condition'].value_counts().to_dict())